# PoE Ninja Character Name Scraper

This notebook scrapes character names from the PoE Ninja website for the specified build, using delays to avoid rate limiting and bot detection.

In [2]:
# Import required libraries
import os
import time
import pandas as pd
from pathlib import Path

from datetime import timedelta

from src.selenium_service import fetch_html_with_selenium, CSSWaitSelector
from src.poeninja.url_service import construct_search_url, SearchParams, TimelessJewels, TimeMachine, SortBy
from src.poeninja.parse_service import parse_search_html

RAW_SEARCH_DIR = "data/poeninja_search/"
PARSED_SEARCH_DIR = "data/poeninja_search_parsed/"

In [3]:
def write_bytes(fpath: str, content: bytes) -> None:
    with open(fpath, "wb") as f:
        f.write(content.encode("utf-8"))

def read_bytes(fpath: str) -> bytes:
    with open(fpath, "rb") as f:
        return f.read()

# Fetch URLS

In [ ]:
url = construct_search_url(
    base_url="https://poe.ninja",
    league="mirage",
    params=SearchParams(
        timeless_jewel=TimelessJewels.LETHAL_PRIDE,
        time_machine=TimeMachine.LATEST,
        sort_by=SortBy.DPS,
        sort_asc=False,
        min_level=90,
        max_life=5000,
    )
)

html_content, elapsed_seconds = fetch_html_with_selenium(
    url=url, 
    timeout=30, 
    selector=CSSWaitSelector.POENINJA_SEARCH,
)

print("Fetching url and waited for page load and content took:", timedelta(seconds=elapsed_seconds))

html_path = os.path.join("data/poeninja_search", f"{int(time.time())}.html")
write_bytes(html_path, html_content)
print(f"Saved HTML to: {html_path}")

# Parse Searches and Consolidate

In [5]:
for f in os.listdir(RAW_SEARCH_DIR):
    html_content = read_bytes(os.path.join(RAW_SEARCH_DIR, f))
    characters, num_characters = parse_search_html(html_content)
    print(f'Parse {len(characters)} characters from {f} (Total from search: {num_characters})')
    pd.DataFrame(characters).to_csv(os.path.join(PARSED_SEARCH_DIR, f"{Path(f).stem}.csv"), index=False)

Parse 100 characters from 1776152340.html (Total from search: 23215)
